In [1]:
import numpy as np
import pandas as pd
import data_management as dm
import feature_builders as fb
import tree_models as tm
import model_wrappers as wp

In [2]:
import plotly.graph_objects as go

def plot_var_vs_loss(var_df, residualbank, title='VaR Forecast vs Realized Loss'):
    """
    VaR Forecast ve Realized Loss'u plotly ile çizer.
    Hover'da predicted_sigma, mu_hat ve standardized_residual gösterilir.

    Parameters
    ----------
    var_df : pd.DataFrame
        Index'i Date olan, 'varForecast' ve 'loss' kolonlarını içeren dataframe.
    residualbank : ResidualBank
        .to_dataframe() metoduyla predicted_sigma, mu_hat, standardized_residual
        kolonlarını Date index'iyle döndüren obje.
    title : str
        Grafik başlığı.

    Returns
    -------
    fig : plotly.graph_objects.Figure
    """
    resid_df = residualbank.to_dataframe()

    merged = var_df.join(
        resid_df[['predicted_sigma', 'mu_hat', 'standardized_residual']],
        how='left'
    )

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=merged.index,
        y=merged['varForecast'],
        mode='lines',
        name='VaR Forecast',
        line=dict(color='red', width=2),
        customdata=merged[['predicted_sigma', 'mu_hat', 'standardized_residual']].values,
        hovertemplate=(
            '<b>%{x}</b><br>' +
            'VaR Forecast: %{y:.2%}<br>' +
            'Predicted Sigma: %{customdata[0]:.2%}<br>' +
            'mu_hat: %{customdata[1]:.2%}<br>' +
            'Standardized Residual: %{customdata[2]:.2f}' +
            '<extra></extra>'
        )
    ))

    fig.add_trace(go.Scatter(
        x=merged.index,
        y=merged['loss'],
        mode='lines+markers',
        name='Realized Loss',
        line=dict(color='steelblue', width=1),
        marker=dict(size=4),
        hovertemplate='<b>%{x}</b><br>Realized Loss: %{y:.2%}<extra></extra>'
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Date',
        yaxis_title='Value',
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        yaxis_tickformat='.1%'
    )

    return fig

In [3]:
TICKERS = ["AAPL", "JPM", "XOM", "KO"]

WEIGHTS = {
    "AAPL": 0.25,
    "JPM": 0.25,
    "XOM": 0.25,
    "KO": 0.25,
}

df = pd.read_csv(
    "./data_general/4_ticker_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)


tickers_data = df[TICKERS].pct_change().dropna()



In [4]:
calendar_last_180_days = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start="2024-06-03",   # Pazartesi, ~6 ay geriye (180 takvim günü civarı)
    train_end="2024-11-29",     # Cuma, Kasım'ın son iş günü
    forecast_start="2024-12-02",  # Pazartesi
    forecast_end="2024-12-31",
)

[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 4 weekday days had no data and were skipped (likely market holidays): 2024-06-19, 2024-07-04, 2024-09-02, 2024-11-28
[INFO] train set: 126 bars, 4 weekday days skipped
[WARNING] [forecast] 1 weekday day had no data and were skipped (likely market holidays): 2024-12-25
[INFO] forecast set: 21 bars, 1 weekday day skipped


In [5]:
training_feature_long, forecast_feature_long = fb.build_long_features(calendar_last_180_days)

[INFO] long train: 504 rows -> 424 after dropna (80 dropped, 15.9%)
[INFO] long forecast: 84 rows -> 84 after dropna (0 dropped, 0.0%)


# Long Features

In [6]:
training_feature_long.X_in_sample_id_cols.tail(8)

,Date,ticker
412,2024-11-26,AAPL
413,2024-11-26,JPM
414,2024-11-26,KO
415,2024-11-26,XOM
416,2024-11-27,AAPL
417,2024-11-27,JPM
418,2024-11-27,KO
419,2024-11-27,XOM


In [7]:
training_feature_long.X_in_sample.tail(8)

,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,real_vol_21,mean_abs_21,skew_21,ewma_vol,market_ewma_vol
412,0.009404,0.009404,0.005807,0.006723,-0.248618,0.008610,0.008002,-1.097144,0.010734,0.008397,-0.056357,0.009933,0.006493
413,-0.001279,0.001279,0.011119,0.009959,-0.474489,0.009360,0.008270,-0.094341,0.028522,0.014797,2.865696,0.021544,0.006493
414,0.002641,0.002641,0.003988,0.006192,0.826057,0.008035,0.006796,-0.605081,0.009395,0.007192,-0.866169,0.008751,0.006493
415,-0.016671,0.016671,0.014826,0.012078,0.003857,0.012165,0.010307,-0.051789,0.012547,0.009427,0.691546,0.012204,0.006493
416,-0.000553,0.000553,0.006445,0.006202,0.012925,0.008774,0.007661,-0.874251,0.010735,0.008368,-0.031253,0.009523,0.006261
417,-0.000720,0.000720,0.008517,0.008203,0.065721,0.009443,0.007674,0.179524,0.028300,0.014282,2.884405,0.020608,0.006261
418,-0.001859,0.001859,0.005357,0.005286,0.526990,0.007950,0.006666,-0.662816,0.008733,0.006487,-1.057657,0.008369,0.006261
419,-0.002628,0.002628,0.012162,0.009754,0.612738,0.011602,0.009640,0.226672,0.012169,0.008903,0.693591,0.011644,0.006261


In [8]:
training_feature_long.y_in_sample.tail()

,target
415,6.905141e-06
416,1.043629e-04
417,7.851509e-08
418,4.457249e-06
419,6.500587e-06


### Last index logic

In [9]:
training_feature_long.X_out_sample_last_index

,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,real_vol_21,mean_abs_21,skew_21,ewma_vol,market_ewma_vol
0,0.010216,0.010216,0.005224,0.007826,-1.060413,0.008412,0.007305,-0.982780,0.010311,0.008127,-0.186424,0.009359,0.006005
1,-0.000280,0.000280,0.007235,0.004953,1.304218,0.009521,0.007408,0.293095,0.028330,0.013973,2.914109,0.019697,0.006005
2,0.002111,0.002111,0.003210,0.003263,0.229986,0.007238,0.006163,-1.054450,0.008643,0.006327,-1.048754,0.008000,0.006005
3,0.002550,0.002550,0.008664,0.007588,-0.410208,0.011671,0.009828,0.169808,0.012106,0.008785,0.611500,0.011161,0.006005


In [10]:
training_feature_long.X_out_sample_last_index_id_cols #

,Date,ticker
0,2024-11-29,AAPL
1,2024-11-29,JPM
2,2024-11-29,KO
3,2024-11-29,XOM


In [11]:
forecast_feature_long.X_forecast_sample.tail()

,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,real_vol_21,mean_abs_21,skew_21,ewma_vol,market_ewma_vol
79,-0.006762,0.006762,0.003988,0.002541,-1.328193,0.008329,0.006501,-0.999001,0.008485,0.006571,-1.414383,0.008239,0.007450
80,-0.007058,0.007058,0.010856,0.009643,0.702219,0.012947,0.010825,-0.291988,0.010245,0.008417,-0.681828,0.010919,0.007241
81,0.001630,0.001630,0.010034,0.007454,0.872998,0.015249,0.011037,-1.015830,0.011732,0.009018,-0.516830,0.013352,0.007241
82,0.003708,0.003708,0.005805,0.004804,0.451346,0.007023,0.005697,0.895839,0.009143,0.007323,0.393393,0.007909,0.007241
83,0.017114,0.017114,0.008821,0.005151,1.448613,0.008627,0.006074,0.547632,0.009651,0.007265,-0.559216,0.009974,0.007241


# Wide Features

In [12]:
training_feature2, forecast_feature2 = fb.build_wide_features(calendar_last_180_days)


[INFO] wide train: 126 bars -> 106 after dropna (20 dropped, 15.9%)
[INFO] wide forecast: 21 bars -> 21 after dropna (0 dropped, 0.0%)


In [13]:
training_feature2.X_in_sample.tail()

,AAPL,JPM,XOM,KO,portfolio_return,AAPL_abs_r,JPM_abs_r,XOM_abs_r,KO_abs_r,portfolio_return_abs_r,...,AAPL_skew_21,JPM_skew_21,XOM_skew_21,KO_skew_21,portfolio_return_skew_21,AAPL_ewma_vol,JPM_ewma_vol,XOM_ewma_vol,KO_ewma_vol,portfolio_return_ewma_vol
Date,,,,,,,,,,,,,,,,,,,,,
2024-11-21,-0.002096,0.016530,0.013381,0.012224,0.010010,0.002096,0.016530,0.013381,0.012224,0.010010,...,0.078089,2.860280,0.867899,-0.465390,1.575668,0.010369,0.024502,0.011579,0.009682,0.007275
2024-11-22,0.005908,0.015485,-0.001148,0.002509,0.005688,0.005908,0.015485,0.001148,0.002509,0.005688,...,-0.014442,2.789165,0.837406,-0.660982,1.406192,0.010009,0.023591,0.011083,0.009285,0.007043
2024-11-25,0.013051,0.007001,-0.014944,0.007196,0.003076,0.013051,0.007001,0.014944,0.007196,0.003076,...,-0.052969,2.774446,0.779617,-0.774493,1.316960,0.010175,0.022499,0.011671,0.009145,0.006724
2024-11-26,0.009404,-0.001279,-0.016671,0.002641,-0.001476,0.009404,0.001279,0.016671,0.002641,0.001476,...,-0.056357,2.865696,0.691546,-0.866169,1.411950,0.009933,0.021544,0.012204,0.008751,0.006493
2024-11-27,-0.000553,-0.000720,-0.002628,-0.001859,-0.001440,0.000553,0.000720,0.002628,0.001859,0.001440,...,-0.031253,2.884405,0.693591,-1.057657,1.655232,0.009523,0.020608,0.011644,0.008369,0.006261


In [14]:
training_feature2.y_in_sample.tail()

,target
Date,
2024-11-21,0.000032
2024-11-22,0.000009
2024-11-25,0.000002
2024-11-26,0.000002
2024-11-27,0.000013


In [15]:
training_feature2.X_out_sample_last_index

,AAPL,JPM,XOM,KO,portfolio_return,AAPL_abs_r,JPM_abs_r,XOM_abs_r,KO_abs_r,portfolio_return_abs_r,...,AAPL_skew_21,JPM_skew_21,XOM_skew_21,KO_skew_21,portfolio_return_skew_21,AAPL_ewma_vol,JPM_ewma_vol,XOM_ewma_vol,KO_ewma_vol,portfolio_return_ewma_vol
Date,,,,,,,,,,,,,,,,,,,,,
2024-11-29,0.010216,-0.00028,0.00255,0.002111,0.003649,0.010216,0.00028,0.00255,0.002111,0.003649,...,-0.186424,2.914109,0.6115,-1.048754,1.562818,0.009359,0.019697,0.011161,0.008,0.006005


# General Long Flow Example with XGBoost Example

In [16]:
WINDOW_SIZE = 20
VAR_CONFIDENCE = 0.95
model = tm.xgboost_train(calendar_last_180_days, training_feature_long, WINDOW_SIZE)

[INFO] usable_train_dates: 96/105 candidate dates survived burn-in (min_periods=29, train_data starts 2024-06-03, burn-in cutoff 2024-07-16, 9 candidate date(s) dropped).


In [17]:
model

In [18]:
model.residual_bank_in_sample.to_dataframe().head()


,actual_return,predicted_variance,predicted_sigma,mu_hat,standardized_residual
Date,,,,,
2024-07-17,0.004583,0.000035,0.005928,0.0,0.773134
2024-07-18,-0.010688,0.000036,0.006029,0.0,-1.772793
2024-07-19,-0.005455,0.000036,0.006025,0.0,-0.905254
2024-07-22,-0.003508,0.000036,0.005976,0.0,-0.587071
2024-07-23,-0.002069,0.000036,0.006019,0.0,-0.343752


In [19]:
model.residual_bank_in_sample.to_dataframe().tail()

,actual_return,predicted_variance,predicted_sigma,mu_hat,standardized_residual
Date,,,,,
2024-11-25,0.003076,0.000043,0.006558,0.0,0.469065
2024-11-26,-0.001476,0.000041,0.006379,0.0,-0.231402
2024-11-27,-0.00144,0.000038,0.006154,0.0,-0.233996
2024-11-29,0.003649,0.000038,0.006163,0.0,0.592144
2024-12-02,-0.003004,0.000039,0.006209,0.0,-0.483831


In [20]:
model.ewma_variance_bank_in_sample.to_dataframe().tail()

returns                               sigma_diag            \
                AAPL       JPM       XOM        KO       AAPL       JPM   
2024-11-25  0.013051  0.007001 -0.014944  0.007196   0.000173  0.000137   
2024-11-26  0.009404 -0.001279 -0.016671  0.002641   0.000173  0.000137   
2024-11-27 -0.000553  -0.00072 -0.002628 -0.001859    0.00016  0.000137   
2024-11-29  0.010216  -0.00028   0.00255  0.002111    0.00016  0.000137   
2024-12-02  0.009523 -0.013896 -0.000932  -0.00671    0.00016  0.000137   

                              portfolio_variance  
                 XOM       KO portfolio_variance  
2024-11-25  0.000173  0.00016           0.000043  
2024-11-26  0.000173  0.00016           0.000041  
2024-11-27  0.000173  0.00016           0.000038  
2024-11-29  0.000173  0.00016           0.000038  
2024-12-02  0.000173  0.00016           0.000039

In [21]:
model.ewma_variance_bank_in_sample.to_dataframe().head()


returns                               sigma_diag            \
                AAPL       JPM       XOM        KO       AAPL       JPM   
2024-07-17 -0.025296  0.015214  0.013788  0.014626   0.000173  0.000173   
2024-07-18 -0.020535  -0.03177  0.009861 -0.000307   0.000173  0.000173   
2024-07-19   0.00058 -0.000952  -0.02298  0.001534   0.000173  0.000173   
2024-07-22  -0.00156  0.002383 -0.006893 -0.007964   0.000173  0.000173   
2024-07-23  0.004688  0.000238 -0.016136  0.002933   0.000173  0.000173   

                              portfolio_variance  
                 XOM       KO portfolio_variance  
2024-07-17  0.000173  0.00016           0.000035  
2024-07-18  0.000173  0.00016           0.000036  
2024-07-19  0.000173  0.00016           0.000036  
2024-07-22  0.000173  0.00016           0.000036  
2024-07-23  0.000173  0.00016           0.000036

## Single Day Prediction

In [22]:
# variance prediction for direct out of sample date, 2024-21

#model.predict_variance_batch_long()

In [23]:
#model.residual_bank_in_sample.empirical_var_for_target()

In [24]:
var_df, residualbank, ewmabank = dm.multi_day_var_from_long_format_model(
        model, calendar_last_180_days, forecast_feature_long, WINDOW_SIZE, VAR_CONFIDENCE,
        model.residual_bank_in_sample, model.ewma_variance_bank_in_sample,
)

TypeError: multi_day_var_from_long_format_model() missing 1 required positional argument: 'initial_ewma_variance_bank'

In [ ]:
var_df

In [ ]:
plot_var_vs_loss(var_df, residualbank)

In [ ]:
WINDOW_SIZE = 20
model_random_forest = tm.random_forest_train(calendar_last_180_days, training_feature_long, WINDOW_SIZE)

var_df, residualbank, ewmabank = dm.multi_day_var_from_long_format_model(
        model_random_forest, calendar_last_180_days, forecast_feature_long, WINDOW_SIZE, VAR_CONFIDENCE,
        model_random_forest.residual_bank_in_sample, model_random_forest.ewma_variance_bank_in_sample,
)


In [ ]:
plot_var_vs_loss(var_df, residualbank)

In [ ]:
WINDOW_SIZE = 90
model_random_forest = tm.random_forest_train(calendar_last_180_days, training_feature_long, WINDOW_SIZE)

var_df, residualbank, ewmabank = dm.multi_day_var_from_long_format_model(
        model_random_forest, calendar_last_180_days, forecast_feature_long, WINDOW_SIZE, VAR_CONFIDENCE,
        model_random_forest.residual_bank_in_sample, model_random_forest.ewma_variance_bank_in_sample,
)

In [ ]:
plot_var_vs_loss(var_df, residualbank)

In [ ]:

calendar_last_360_days = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start="2023-11-29",
    train_end="2024-11-29",
    forecast_start="2024-12-02",  # Pazartesi
    forecast_end="2024-12-31",
)

In [ ]:
WINDOW_SIZE = 90
model_random_forest = tm.random_forest_train(calendar_last_360_days, training_feature_long, WINDOW_SIZE)


var_df, residualbank, ewmabank = dm.multi_day_var_from_long_format_model(
        model_random_forest, calendar_last_360_days, forecast_feature_long, WINDOW_SIZE, VAR_CONFIDENCE,
        model_random_forest.residual_bank_in_sample, model_random_forest.ewma_variance_bank_in_sample,
)


In [ ]:
plot_var_vs_loss(var_df, residualbank)


In [ ]:
WINDOW_SIZE = 200
model_random_forest = tm.random_forest_train(calendar_last_360_days, training_feature_long, WINDOW_SIZE)


var_df, residualbank, ewmabank = dm.multi_day_var_from_long_format_model(
        model_random_forest, calendar_last_360_days, forecast_feature_long, WINDOW_SIZE, VAR_CONFIDENCE,
        model_random_forest.residual_bank_in_sample, model_random_forest.ewma_variance_bank_in_sample,
)


# General Wide Flow Example with XGBoost Example


In [ ]:
model_wide = tm.xgboost_train(calendar_last_180_days, training_feature2, WINDOW_SIZE)



In [ ]:
model_wide.residual_bank_in_sample.to_dataframe().head()


In [ ]:
model_wide.residual_bank_in_sample.to_dataframe().tail()